# WAS-Mamba — Colab Smoke Test

Purpose: confirm the base-model code actually runs on a real GPU (dependencies install, architecture builds, a forward pass completes) — **no BraTS data needed for this notebook.**

Before running: **Runtime -> Change runtime type -> T4 GPU**, then Runtime -> Run all.

## 1. Get the code

If the repo is **public**, the plain clone below just works.
If it's **private**, run the token cell instead (get a token from github.com -> Settings -> Developer settings -> Personal access tokens -> generate one with `repo` scope; paste it when prompted, it is not saved anywhere).

In [ ]:
# Option A -- public repo
!git clone https://github.com/sachinn854/Brain-Tumor-Segmentatiton.git
%cd Brain-Tumor-Segmentatiton

In [ ]:
# Option B -- private repo (skip Option A above if you use this instead)
import getpass
token = getpass.getpass('GitHub personal access token: ')
!git clone https://{token}@github.com/sachinn854/Brain-Tumor-Segmentatiton.git
%cd Brain-Tumor-Segmentatiton
del token

## 2. Check GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Install dependencies

`torch` is already on Colab. `mamba-ssm`'s CUDA kernels take a few minutes to build here the first time -- that's expected, not a hang.

In [ ]:
!pip install -q einops timm nibabel
!pip install -q causal-conv1d>=1.4.0
!pip install -q mamba-ssm

## 4. Import the model

This alone catches import-time bugs (e.g. the missing `to_3tuple` import that was in the original repo's file and got fixed in `src/models/wasmamba.py`).

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

from src.models.wasmamba import WASMamba
print('Model imported OK')

## 5. Build the model with BraTS-shaped settings

`input_channels=4` (FLAIR, T1w, T1Gd, T2w), `num_classes=4` (background + 3 tumor classes) -- matches `src/configs/wasmamba_config.py`.

In [ ]:
model = WASMamba(
    input_channels=4,
    num_classes=4,
    depths=[1, 1, 1, 1],
    depths_decoder=[1, 1, 1, 1],
    drop_path_rate=0.2,
).cuda()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model built OK -- {n_params/1e6:.2f}M parameters')

## 6. Forward pass on a small dummy volume

Using a small 32^3 crop here (not the paper's 128^3) on purpose -- this cell is only checking "does the code run without crashing", not fitting the real training config. The 128^3 / batch=2 config is what actually goes in `src/configs/wasmamba_config.py` for real training.

In [ ]:
dummy_input = torch.randn(1, 4, 32, 32, 32).cuda()

with torch.no_grad():
    output = model(dummy_input)

print('Input shape: ', dummy_input.shape)
print('Output shape:', output.shape)
print('Forward pass OK -- architecture runs end-to-end on GPU')

## 7. Sanity-check the loss function too

In [ ]:
from src.losses.losses import PaperDiceCeLoss

criterion = PaperDiceCeLoss(num_classes=4)
dummy_label = torch.randint(0, 4, (1, 32, 32, 32)).cuda()

loss = criterion(output, dummy_label)
print('Loss value:', loss.item())
print('Loss function OK')

---
If every cell above ran without error: the base model, its loss, and the GPU environment are all confirmed working. Next step is plugging in real BraTS data (`src/data/brats_dataset.py`) once it's downloaded and on Drive -- not part of this notebook.